In [1]:
pip install requests beautifulsoup4 pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install selenium webdriver-manager

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 9.7/9.7 MB 50.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.2.3
    Uninstalling urllib3-2.2.3:
      Successfully uninstalled urllib3-2.2.3
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: certifi
    Found existing installation: certifi 2024.8.30
    Uninstalling certifi-2024.8.30:
      Successfully uninstalled certifi-2024.8.30
  Attempting uninstall: attrs
    Found existing installation: attrs 23.1.0
    Uninstalling attrs-23.1.0:
      Successfully uninstalled attrs-23.1.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.32.0 which is incompatible.


In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# oliveyoung_skincare_cleaned.py
# pip install selenium webdriver-manager beautifulsoup4 pandas openpyxl

import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# ====== 기본 설정 ======
TARGET_URL = "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do?dispCatNo=100000100010013&isLoginCnt=0&aShowCnt=0&bShowCnt=0&cShowCnt=0&gateCd=Drawer&trackingCd=Cat100000100010013_MID&trackingCd=Cat100000100010013_MID&t_page=%EB%93%9C%EB%A1%9C%EC%9A%B0_%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC&t_click=%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC%ED%83%AD_%EC%A4%91%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC&t_1st_category_type=%EB%8C%80_%EC%8A%A4%ED%82%A8%EC%BC%80%EC%96%B4&t_2nd_category_type=%EC%A4%91_%EC%8A%A4%ED%82%A8%2F%ED%86%A0%EB%84%88"
OUTPUT_FILE = "oliveyoung_skincare_top10_cleaned.xlsx"
HEADLESS = False   # True로 하면 브라우저 창 없이 실행
WAIT_TIME = 3
# =========================


def start_driver():
    """Selenium 크롬 드라이버 실행"""
    options = Options()
    if HEADLESS:
        options.add_argument("--headless=new")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)

    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def parse_products(html):
    """BeautifulSoup으로 상품 정보 추출"""
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    # 공통 상품 카드 구조 탐색
    products = soup.select("ul.cate_prd_list li")
    if not products:
        products = soup.select("div.prd_info")

    for p in products:
        name_tag = p.select_one(".prd_name")
        price_tag = p.select_one(".price span")
        link_tag = p.select_one("a[href]")

        name = name_tag.get_text(strip=True) if name_tag else None
        price = price_tag.get_text(strip=True) if price_tag else None
        link = link_tag["href"] if link_tag else None

        if link and link.startswith("/"):
            link = "https://www.oliveyoung.co.kr" + link

        if name:
            rows.append({"name": name, "price": price, "link": link})

    return rows


def clean_product_name(name: str):
    """불필요한 문구 제거하고 핵심 제품명만 남김"""
    name = re.sub(r"\[.*?\]|\(.*?\)|\+.*", "", name)   # 괄호, 증정 문구 제거
    name = re.sub(r"(기획|세트|한정|증정|올영픽|프로모션)", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def extract_category(name: str):
    """제품명에서 화장품 종류 추출"""
    categories = ["토너", "크림", "로션", "앰플", "세럼", "미스트", "패드", "클렌징", "에센스", "마스크"]
    for c in categories:
        if c in name:
            return c
    return "기타"


def main():
    driver = start_driver()
    try:
        print("📂 올리브영 스킨케어 페이지 접속 중...")
        driver.get(TARGET_URL)
        time.sleep(WAIT_TIME)

        html = driver.page_source
        items = parse_products(html)

        # 🔍 스킨케어 관련 품목만 필터링
        skincare_keywords = ["스킨", "토너", "로션", "크림", "앰플", "세럼", "패드", "미스트"]
        skincare_items = [i for i in items if any(k in i["name"] for k in skincare_keywords)]

        # ✨ 이름 정제 및 카테고리 추가
        for i in skincare_items:
            i["product_name"] = clean_product_name(i["name"])
            i["category"] = extract_category(i["product_name"])

        # 🔢 상위 10개만 추출
        top10 = skincare_items[:10]

        if not top10:
            print("❌ 스킨케어 품목을 찾지 못했습니다. (사이트 구조 변경 가능)")
            return

        df = pd.DataFrame(top10)
        df = df[["product_name", "category", "price", "link"]]  # 컬럼 순서 정리
        df.to_excel(OUTPUT_FILE, index=False)

        print(f"\n✅ 상위 10개 스킨케어 상품을 '{OUTPUT_FILE}'로 저장했습니다!\n")
        print(df)

    finally:
        driver.quit()
        print("\n브라우저 종료 완료.")


if __name__ == "__main__":
    main()


📂 올리브영 스킨케어 페이지 접속 중...

✅ 상위 10개 스킨케어 상품을 'oliveyoung_skincare_top10_cleaned.xlsx'로 저장했습니다!

                               product_name category price  \
0               바이오더마 바이오더마 하이드라비오 토너 500ml       토너  None   
1      브링그린 피지쓱싹 브링그린 티트리시카수딩토너 250mL/500mL       토너  None   
2                     라네즈 라네즈 크림스킨 170ml 리필       크림  None   
3                아누아 아누아 어성초 77 수딩 토너 350ml       토너  None   
4  더랩바이블랑두 더랩바이블랑두 올리고 히알루론산 딥 토너 500ml 대용량       토너  None   
5                라운드랩 라운드랩 1025 독도 토너 300ml       토너  None   
6        넘버즈인 넘버즈인 3번 결광가득 에센스 토너 300ml 대용량       토너  None   
7             웰라쥬 웰라쥬 리얼 히알루로닉 100 토너 300ml       토너  None   
8         넘버즈인넘버즈인 1번 진정 맑게담은 청초토너 300ml 리필       토너  None   
9             에스네이처 에스네이처 아쿠아 오아시스 토너 300ml       토너  None   

                                                link  
0  https://www.oliveyoung.co.kr/store/goods/getGo...  
1  https://www.oliveyoung.co.kr/store/goods/getGo...  
2  https://www.oliveyoung.co.kr/store/goods/getGo...  
3  

In [9]:
import os
print("📂 저장 경로:", os.path.abspath(OUTPUT_FILE))


📂 저장 경로: C:\Users\UserK\oliveyoung_skincare_top10_cleaned.xlsx


In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# pip install selenium webdriver-manager beautifulsoup4 pandas openpyxl

import time
import re
import os
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ===== 설정 =====
TARGET_URL = "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do?dispCatNo=100000100010013"
OUTPUT_FILE = "oliveyoung_skincare_top10_with_ingredients.xlsx"
HEADLESS = False
WAIT_TIME = 5
# ================


def start_driver():
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1200,1000")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option('useAutomationExtension', False)
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)


def parse_products(html):
    """카테고리 페이지에서 상품명/링크 추출"""
    soup = BeautifulSoup(html, "html.parser")
    products = []

    cards = soup.select("ul.cate_prd_list li") or soup.select("div.prd_info")

    for c in cards:
        name_tag = c.select_one(".prd_name")
        link_tag = c.select_one("a[href]")
        name = name_tag.get_text(strip=True) if name_tag else None
        link = link_tag["href"] if link_tag else None
        if link and link.startswith("/"):
            link = "https://www.oliveyoung.co.kr" + link
        if name:
            products.append({"name": name, "link": link})
    return products


def clean_product_name(name: str):
    """불필요 문구 제거"""
    name = re.sub(r"\[.*?\]|\(.*?\)|\+.*", "", name)
    name = re.sub(r"(기획|세트|한정|증정|올영픽|프로모션)", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def extract_category(name: str):
    """제품명에서 화장품 종류 추출"""
    categories = ["토너", "크림", "로션", "앰플", "세럼", "미스트", "패드", "클렌징", "에센스", "마스크"]
    for c in categories:
        if c in name:
            return c
    return "기타"


def extract_ingredients(driver, product_url):
    """
    상품 상세 페이지에서 전성분 추출
    """
    try:
        driver.get(product_url)
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "body"))
        )
        time.sleep(1.5)  # 상세 정보 로딩 대기

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        # 전성분 영역 여러 형태로 시도
        selectors = [
            "div.prd_detail_info",
            "div#artcInfo",
            "div.information",
            "div#productDetail",
            "li:contains('전성분')",
        ]

        for sel in selectors:
            tag = soup.select_one(sel)
            if tag and "전성분" in tag.get_text():
                text = tag.get_text(" ", strip=True)
                # "전성분" 이후의 내용만 추출
                match = re.search(r"전성분[:：]?\s*(.*)", text)
                if match:
                    return match.group(1).split("기능성")[
                        0
                    ]  # 기능성 안내 등 앞부분만
        # 다른 형태의 label 시도
        labels = soup.find_all(text=re.compile("전성분|Ingredients", re.I))
        if labels:
            for label in labels:
                parent = label.find_parent()
                if parent:
                    text = parent.get_text(" ", strip=True)
                    match = re.search(r"전성분[:：]?\s*(.*)", text)
                    if match:
                        return match.group(1)
        return "성분 정보 없음"
    except Exception:
        return "성분 정보 없음"


def main():
    driver = start_driver()
    try:
        print("📂 스킨케어 목록 로드 중...")
        driver.get(TARGET_URL)
        WebDriverWait(driver, WAIT_TIME).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".prd_name"))
        )
        time.sleep(2)

        html = driver.page_source
        items = parse_products(html)

        # 스킨케어 관련 키워드만
        skincare_keywords = ["스킨", "토너", "로션", "크림", "앰플", "세럼", "패드", "미스트"]
        skincare_items = [i for i in items if any(k in i["name"] for k in skincare_keywords)]

        # 상위 10개만
        skincare_items = skincare_items[:10]

        enriched = []
        for i, item in enumerate(skincare_items, start=1):
            print(f"({i}/{len(skincare_items)}) 🔍 {item['name']} 성분 수집 중...")
            clean_name = clean_product_name(item["name"])
            category = extract_category(clean_name)
            ingredients = extract_ingredients(driver, item["link"])
            enriched.append({
                "product_name": clean_name,
                "category": category,
                "ingredients": ingredients,
                "link": item["link"]
            })
            print(f" → 완료: {clean_name[:25]}...")

        df = pd.DataFrame(enriched)
        df.to_excel(OUTPUT_FILE, index=False)
        print(f"\n✅ 엑셀 저장 완료: {os.path.abspath(OUTPUT_FILE)}")
        print(df.head())

    finally:
        driver.quit()
        print("브라우저 종료 완료.")


if __name__ == "__main__":
    main()


📂 스킨케어 목록 로드 중...
(1/10) 🔍 바이오더마[11월 올영픽] 바이오더마 하이드라비오 토너 500ml 기획(+화장솜 20매 증정) 성분 수집 중...


C:\ProgramData\anaconda3\Lib\site-packages\soupsieve\css_parser.py:856: FutureWarning: The pseudo class ':contains' is deprecated, ':-soup-contains' should be used moving forward.
  warnings.warn(  # noqa: B028
C:\Users\UserK\AppData\Local\Temp\ipykernel_7944\3553724025.py:109: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  labels = soup.find_all(text=re.compile("전성분|Ingredients", re.I))


 → 완료: 바이오더마 바이오더마 하이드라비오 토너 500...
(2/10) 🔍 브링그린[블프|일주일특가] 피지쓱싹 브링그린 티트리시카수딩토너 250mL/500mL 기획 성분 수집 중...
 → 완료: 브링그린 피지쓱싹 브링그린 티트리시카수딩토너 ...
(3/10) 🔍 라네즈[미스트 기획/화잘먹] 라네즈 크림스킨 170ml 리필기획 (+170ml 리필+50ml+미스트펌프) 성분 수집 중...
 → 완료: 라네즈 라네즈 크림스킨 170ml 리필...
(4/10) 🔍 아누아[대용량 기획] 아누아 어성초 77 수딩 토너 350ml 기획 (+350ml 리필팩) 성분 수집 중...
 → 완료: 아누아 아누아 어성초 77 수딩 토너 350m...
(5/10) 🔍 더랩바이블랑두[속보습] 더랩바이블랑두 올리고 히알루론산 딥 토너 500ml 대용량 기획 (+100ml) 성분 수집 중...
 → 완료: 더랩바이블랑두 더랩바이블랑두 올리고 히알루론산...
(6/10) 🔍 라운드랩[한정] 라운드랩 1025 독도 토너 300ml 기획 (+100ml+늘어나는 스킨 패드 30매) 성분 수집 중...
 → 완료: 라운드랩 라운드랩 1025 독도 토너 300m...
(7/10) 🔍 넘버즈인[모공개선/탄력광채] 넘버즈인 3번 결광가득 에센스 토너 300ml 대용량 기획 성분 수집 중...
 → 완료: 넘버즈인 넘버즈인 3번 결광가득 에센스 토너 ...
(8/10) 🔍 웰라쥬[첫수분토너/대용량] 웰라쥬 리얼 히알루로닉 100 토너 300ml 기획 (+화장솜 60매) 성분 수집 중...
 → 완료: 웰라쥬 웰라쥬 리얼 히알루로닉 100 토너 3...
(9/10) 🔍 넘버즈인[쿨링진정]넘버즈인 1번 진정 맑게담은 청초토너 300ml 리필기획(+300ml 증정) 성분 수집 중...
 → 완료: 넘버즈인넘버즈인 1번 진정 맑게담은 청초토너 ...
(10/10) 🔍 에스네이처[수분진정/화해1위] 에스네이처 아쿠아 오아시스 토너 300ml 기획 (+수분크림 30ml) 성분 수집 중...
 → 완료: 에스

In [11]:
import time, re, os
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ===== 설정 =====
TARGET_URL = "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do?dispCatNo=100000100010013"
OUTPUT_FILE = "oliveyoung_skincare_main_ingredient.xlsx"
HEADLESS = False
WAIT_TIME = 5
# =================

MAIN_INGREDIENTS = [
    "히알루론산", "시카", "센텔라", "판테놀", "비타민", "나이아신아마이드", "레티놀",
    "세라마이드", "콜라겐", "프로폴리스", "녹차", "알로에", "병풀", "AHA", "PHA",
    "BHA", "마데카소사이드", "달팽이", "펩타이드"
]


def start_driver():
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1200,1000")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option('useAutomationExtension', False)
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)


def parse_products(html):
    soup = BeautifulSoup(html, "html.parser")
    items = []
    for li in soup.select("ul.cate_prd_list li"):
        name_tag = li.select_one(".prd_name")
        link_tag = li.select_one("a[href]")
        if not name_tag or not link_tag:
            continue
        name = name_tag.get_text(strip=True)
        link = link_tag["href"]
        if link.startswith("/"):
            link = "https://www.oliveyoung.co.kr" + link
        items.append({"name": name, "link": link})
    return items


def clean_product_name(name):
    name = re.sub(r"\[.*?\]|\(.*?\)|\+.*", "", name)
    name = re.sub(r"(기획|세트|한정|증정|올영픽|프로모션)", "", name)
    return name.strip()


def extract_category(name):
    for c in ["토너", "크림", "로션", "앰플", "세럼", "미스트", "패드", "클렌징", "에센스", "마스크"]:
        if c in name:
            return c
    return "기타"


def extract_main_ingredient(driver, url):
    """상세 페이지에서 주요 성분 키워드 중 첫 번째 발견"""
    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "body")))
        time.sleep(1.5)

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        text = soup.get_text(" ", strip=True)

        for ing in MAIN_INGREDIENTS:
            if ing in text:
                return ing
        return "주요 성분 정보 없음"
    except Exception:
        return "에러 발생"


def main():
    driver = start_driver()
    try:
        driver.get(TARGET_URL)
        WebDriverWait(driver, WAIT_TIME).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".prd_name")))
        time.sleep(2)

        html = driver.page_source
        items = parse_products(html)

        skincare_keywords = ["스킨", "토너", "로션", "크림", "앰플", "세럼", "패드", "미스트"]
        skincare_items = [i for i in items if any(k in i["name"] for k in skincare_keywords)][:10]

        enriched = []
        for idx, item in enumerate(skincare_items, start=1):
            print(f"({idx}/10) {item['name']} → 주요 성분 추출 중...")
            clean_name = clean_product_name(item["name"])
            category = extract_category(clean_name)
            main_ing = extract_main_ingredient(driver, item["link"])
            enriched.append({
                "product_name": clean_name,
                "category": category,
                "main_ingredient": main_ing,
                "link": item["link"]
            })

        df = pd.DataFrame(enriched)
        df.to_excel(OUTPUT_FILE, index=False)
        print(f"\n✅ 엑셀 저장 완료: {os.path.abspath(OUTPUT_FILE)}")
        print(df)
    finally:
        driver.quit()
        print("브라우저 종료 완료.")


if __name__ == "__main__":
    main()


(1/10) 바이오더마[11월 올영픽] 바이오더마 하이드라비오 토너 500ml 기획(+화장솜 20매 증정) → 주요 성분 추출 중...
(2/10) 브링그린[블프|일주일특가] 피지쓱싹 브링그린 티트리시카수딩토너 250mL/500mL 기획 → 주요 성분 추출 중...
(3/10) 라네즈[미스트 기획/화잘먹] 라네즈 크림스킨 170ml 리필기획 (+170ml 리필+50ml+미스트펌프) → 주요 성분 추출 중...
(4/10) 아누아[대용량 기획] 아누아 어성초 77 수딩 토너 350ml 기획 (+350ml 리필팩) → 주요 성분 추출 중...
(5/10) 라운드랩[한정] 라운드랩 1025 독도 토너 300ml 기획 (+100ml+늘어나는 스킨 패드 30매) → 주요 성분 추출 중...
(6/10) 넘버즈인[모공개선/탄력광채] 넘버즈인 3번 결광가득 에센스 토너 300ml 대용량 기획 → 주요 성분 추출 중...
(7/10) 웰라쥬[첫수분토너/대용량] 웰라쥬 리얼 히알루로닉 100 토너 300ml 기획 (+화장솜 60매) → 주요 성분 추출 중...
(8/10) 넘버즈인[쿨링진정]넘버즈인 1번 진정 맑게담은 청초토너 300ml 리필기획(+300ml 증정) → 주요 성분 추출 중...
(9/10) 에스네이처[수분진정/화해1위] 에스네이처 아쿠아 오아시스 토너 300ml 기획 (+수분크림 30ml) → 주요 성분 추출 중...
(10/10) 라운드랩[블프특가/9,900원] 라운드랩 포 맨 자작나무 수분 토너 200ml → 주요 성분 추출 중...

✅ 엑셀 저장 완료: C:\Users\UserK\oliveyoung_skincare_main_ingredient.xlsx
                           product_name category main_ingredient  \
0           바이오더마 바이오더마 하이드라비오 토너 500ml       토너              시카   
1  브링그린 피지쓱싹 브링그린 티트리시카수딩토너 250mL/5

In [12]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# pip install selenium webdriver-manager beautifulsoup4 pandas openpyxl

import time, re, os, pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ===== 설정 =====
TARGET_URL = "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do?dispCatNo=100000100010013"
OUTPUT_FILE = "oliveyoung_skincare_ad_ingredients.xlsx"
HEADLESS = False
WAIT_TIME = 5
# 광고 문구에서 찾을 주요 성분 키워드
TARGET_PHRASES = ["다프", "DAF", "아쿠아지니움", "Aquagenium", "시카", "히알루론산", "콜라겐"]
# =================

def start_driver():
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1200,1000")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option('useAutomationExtension', False)
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)


def parse_products(html):
    """카테고리 페이지에서 상품명/링크 추출"""
    soup = BeautifulSoup(html, "html.parser")
    products = []
    for li in soup.select("ul.cate_prd_list li"):
        name_tag = li.select_one(".prd_name")
        link_tag = li.select_one("a[href]")
        if not name_tag or not link_tag:
            continue
        name = name_tag.get_text(strip=True)
        link = link_tag["href"]
        if link.startswith("/"):
            link = "https://www.oliveyoung.co.kr" + link
        products.append({"name": name, "link": link})
    return products


def clean_product_name(name):
    name = re.sub(r"\[.*?\]|\(.*?\)|\+.*", "", name)
    name = re.sub(r"(기획|세트|한정|증정|올영픽|프로모션)", "", name)
    return name.strip()


def extract_category(name):
    for c in ["토너", "크림", "로션", "앰플", "세럼", "미스트", "패드", "클렌징", "에센스", "마스크"]:
        if c in name:
            return c
    return "기타"


def extract_highlight_phrase(driver, url):
    """
    상세 페이지에서 광고 문구 중 특정 성분 키워드가 포함된 문장 추출
    """
    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "body")))
        time.sleep(1.5)
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        text = soup.get_text(" ", strip=True)

        # 상품 상세 설명 영역(전성분 말고 광고문구 쪽) 찾기
        desc_section = ""
        for sel in ["div.prd_detail_info", "div#artcInfo", "div.information", "div#productDetail", "div.cont", "div.txt"]:
            section = soup.select_one(sel)
            if section and len(section.get_text(strip=True)) > 100:
                desc_section = section.get_text(" ", strip=True)
                break

        if not desc_section:
            desc_section = text  # fallback

        # 문장 단위 분리
        sentences = re.split(r"[.!?…\n]", desc_section)
        for s in sentences:
            for key in TARGET_PHRASES:
                if re.search(key, s, re.IGNORECASE):
                    return s.strip()
        return "광고 문구에 해당 성분 언급 없음"

    except Exception as e:
        return f"오류: {e}"


def main():
    driver = start_driver()
    try:
        print("📂 스킨케어 목록 로드 중...")
        driver.get(TARGET_URL)
        WebDriverWait(driver, WAIT_TIME).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".prd_name")))
        time.sleep(2)

        html = driver.page_source
        items = parse_products(html)

        # 스킨케어 키워드 필터링
        skincare_keywords = ["스킨", "토너", "로션", "크림", "앰플", "세럼", "패드", "미스트"]
        skincare_items = [i for i in items if any(k in i["name"] for k in skincare_keywords)][:10]

        enriched = []
        for idx, item in enumerate(skincare_items, start=1):
            print(f"({idx}/{len(skincare_items)}) {item['name']} → 광고 성분 문구 추출 중...")
            clean_name = clean_product_name(item["name"])
            category = extract_category(clean_name)
            highlight = extract_highlight_phrase(driver, item["link"])
            enriched.append({
                "product_name": clean_name,
                "category": category,
                "highlight_phrase": highlight,
                "link": item["link"]
            })

        df = pd.DataFrame(enriched)
        df.to_excel(OUTPUT_FILE, index=False)
        print(f"\n✅ 엑셀 저장 완료: {os.path.abspath(OUTPUT_FILE)}")
        print(df)
    finally:
        driver.quit()
        print("브라우저 종료 완료.")


if __name__ == "__main__":
    main()


📂 스킨케어 목록 로드 중...
(1/10) 바이오더마[11월 올영픽] 바이오더마 하이드라비오 토너 500ml 기획(+화장솜 20매 증정) → 광고 성분 문구 추출 중...
(2/10) 브링그린[블프|일주일특가] 피지쓱싹 브링그린 티트리시카수딩토너 250mL/500mL 기획 → 광고 성분 문구 추출 중...
(3/10) 라네즈[미스트 기획/화잘먹] 라네즈 크림스킨 170ml 리필기획 (+170ml 리필+50ml+미스트펌프) → 광고 성분 문구 추출 중...
(4/10) 아누아[대용량 기획] 아누아 어성초 77 수딩 토너 350ml 기획 (+350ml 리필팩) → 광고 성분 문구 추출 중...
(5/10) 라운드랩[한정] 라운드랩 1025 독도 토너 300ml 기획 (+100ml+늘어나는 스킨 패드 30매) → 광고 성분 문구 추출 중...
(6/10) 넘버즈인[모공개선/탄력광채] 넘버즈인 3번 결광가득 에센스 토너 300ml 대용량 기획 → 광고 성분 문구 추출 중...
(7/10) 웰라쥬[첫수분토너/대용량] 웰라쥬 리얼 히알루로닉 100 토너 300ml 기획 (+화장솜 60매) → 광고 성분 문구 추출 중...
(8/10) 넘버즈인[쿨링진정]넘버즈인 1번 진정 맑게담은 청초토너 300ml 리필기획(+300ml 증정) → 광고 성분 문구 추출 중...
(9/10) 에스네이처[수분진정/화해1위] 에스네이처 아쿠아 오아시스 토너 300ml 기획 (+수분크림 30ml) → 광고 성분 문구 추출 중...
(10/10) 라운드랩[블프특가/9,900원] 라운드랩 포 맨 자작나무 수분 토너 200ml → 광고 성분 문구 추출 중...

✅ 엑셀 저장 완료: C:\Users\UserK\oliveyoung_skincare_ad_ingredients.xlsx
                           product_name category  \
0           바이오더마 바이오더마 하이드라비오 토너 500ml       토너   
1  브링그린 피지쓱싹 브링그린 티트

In [14]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# pip install selenium webdriver-manager beautifulsoup4 pandas openpyxl

import time, re, os, pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ===== 설정 =====
TARGET_URL = "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do?dispCatNo=100000100010013"
OUTPUT_FILE = "oliveyoung_ad_ingredient_recommend.xlsx"
HEADLESS = False
WAIT_TIME = 5

# 광고성 주요 성분 키워드 (브랜드 고유 기술 포함)
AD_KEYWORDS = {
    "다프": "피부 자체의 방어력을 길러주는 활성 성분 복합체",
    "DAF": "피부 자체의 방어력을 길러주는 활성 성분 복합체",
    "아쿠아지니움": "수분 손실 방지 및 수분 공급 강화 복합체",
    "Aquagenium": "수분 손실 방지 및 수분 공급 강화 복합체",
    "시카": "피부 진정 성분",
    "히알루론산": "보습 강화 성분",
    "콜라겐": "탄력 강화 성분"
}
# =================


def start_driver():
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1200,1000")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option('useAutomationExtension', False)
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)


def parse_products(html):
    soup = BeautifulSoup(html, "html.parser")
    products = []
    for li in soup.select("ul.cate_prd_list li"):
        name_tag = li.select_one(".prd_name")
        link_tag = li.select_one("a[href]")
        if not name_tag or not link_tag:
            continue
        name = name_tag.get_text(strip=True)
        link = link_tag["href"]
        if link.startswith("/"):
            link = "https://www.oliveyoung.co.kr" + link
        products.append({"name": name, "link": link})
    return products


def clean_product_name(name):
    name = re.sub(r"\[.*?\]|\(.*?\)|\+.*", "", name)
    name = re.sub(r"(기획|세트|한정|증정|올영픽|프로모션)", "", name)
    return name.strip()


def extract_category(name):
    for c in ["토너", "크림", "로션", "앰플", "세럼", "미스트", "패드", "클렌징", "에센스", "마스크"]:
        if c in name:
            return c
    return "기타"


def extract_ad_ingredient(driver, url):
    """
    상품 상세 설명 영역에서 광고용 주요 성분 문구 추출
    """
    try:
        driver.get(url)
        WebDriverWait(driver, 30).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "body")))
        time.sleep(1.5)

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        text = soup.get_text(" ", strip=True)

        # 광고 문구가 주로 들어가는 섹션
        desc_section = ""
        for sel in ["div.prd_detail_info", "div#artcInfo", "div#productDetail", "div.cont", "div.txt"]:
            section = soup.select_one(sel)
            if section and len(section.get_text(strip=True)) > 50:
                desc_section = section.get_text(" ", strip=True)
                break
        if not desc_section:
            desc_section = text

        # 성분 탐지
        found = []
        for key in AD_KEYWORDS.keys():
            if re.search(key, desc_section, re.IGNORECASE):
                found.append(key)

        if found:
            matched = [f"{k} - {AD_KEYWORDS[k]}" for k in found]
            return " / ".join(matched)
        else:
            return "광고 성분 없음"

    except Exception as e:
        return f"오류: {e}"


def main():
    driver = start_driver()
    try:
        print("📂 스킨케어 목록 로드 중...")
        driver.get(TARGET_URL)
        WebDriverWait(driver, WAIT_TIME).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".prd_name")))
        time.sleep(2)

        html = driver.page_source
        items = parse_products(html)

        skincare_keywords = ["스킨", "토너", "로션", "크림", "앰플", "세럼", "패드", "미스트"]
        skincare_items = [i for i in items if any(k in i["name"] for k in skincare_keywords)][:10]

        enriched = []
        for idx, item in enumerate(skincare_items, start=1):
            print(f"({idx}/10) {item['name']} → 광고 성분 추출 중...")
            clean_name = clean_product_name(item["name"])
            category = extract_category(clean_name)
            ad_ing = extract_ad_ingredient(driver, item["link"])
            enriched.append({
                "product_name": clean_name,
                "category": category,
                "highlight_ad_ingredient": ad_ing,
                "link": item["link"]
            })

        df = pd.DataFrame(enriched)
        df.to_excel(OUTPUT_FILE, index=False)
        print(f"\n✅ 엑셀 저장 완료: {os.path.abspath(OUTPUT_FILE)}")
        print(df)
    finally:
        driver.quit()
        print("브라우저 종료 완료.")


if __name__ == "__main__":
    main()


📂 스킨케어 목록 로드 중...
(1/10) 바이오더마[11월 올영픽] 바이오더마 하이드라비오 토너 500ml 기획(+화장솜 20매 증정) → 광고 성분 추출 중...
(2/10) 브링그린[블프|일주일특가] 피지쓱싹 브링그린 티트리시카수딩토너 250mL/500mL 기획 → 광고 성분 추출 중...
(3/10) 라네즈[미스트 기획/화잘먹] 라네즈 크림스킨 170ml 리필기획 (+170ml 리필+50ml+미스트펌프) → 광고 성분 추출 중...
(4/10) 아누아[대용량 기획] 아누아 어성초 77 수딩 토너 350ml 기획 (+350ml 리필팩) → 광고 성분 추출 중...
(5/10) 넘버즈인[모공개선/탄력광채] 넘버즈인 3번 결광가득 에센스 토너 300ml 대용량 기획 → 광고 성분 추출 중...
(6/10) 라운드랩[한정] 라운드랩 1025 독도 토너 300ml 기획 (+100ml+늘어나는 스킨 패드 30매) → 광고 성분 추출 중...
(7/10) 웰라쥬[첫수분토너/대용량] 웰라쥬 리얼 히알루로닉 100 토너 300ml 기획 (+화장솜 60매) → 광고 성분 추출 중...
(8/10) 넘버즈인[쿨링진정]넘버즈인 1번 진정 맑게담은 청초토너 300ml 리필기획(+300ml 증정) → 광고 성분 추출 중...
(9/10) 라로슈포제라로슈포제 시카플라스트 로션 B5 판테놀 시카에센스 토너 200ml → 광고 성분 추출 중...
(10/10) 에스네이처[수분진정/화해1위] 에스네이처 아쿠아 오아시스 토너 300ml 기획 (+수분크림 30ml) → 광고 성분 추출 중...
브라우저 종료 완료.


PermissionError: [Errno 13] Permission denied: 'oliveyoung_ad_ingredient_recommend.xlsx'

In [15]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# pip install selenium webdriver-manager beautifulsoup4 pandas openpyxl

import time, re, os, pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ===== 설정 =====
TARGET_URL = "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do?dispCatNo=100000100010013"
OUTPUT_FILE = "oliveyoung_top_ingredients.xlsx"
HEADLESS = False
WAIT_TIME = 5
NUM_INGREDIENTS = 10   # 전성분 중 앞부분 몇 개 추출할지
# =================


def start_driver():
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1200,1000")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option('useAutomationExtension', False)
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)


def parse_products(html):
    """카테고리 페이지에서 상품명/링크 추출"""
    soup = BeautifulSoup(html, "html.parser")
    products = []
    for li in soup.select("ul.cate_prd_list li"):
        name_tag = li.select_one(".prd_name")
        link_tag = li.select_one("a[href]")
        if not name_tag or not link_tag:
            continue
        name = name_tag.get_text(strip=True)
        link = link_tag["href"]
        if link.startswith("/"):
            link = "https://www.oliveyoung.co.kr" + link
        products.append({"name": name, "link": link})
    return products


def clean_product_name(name):
    name = re.sub(r"\[.*?\]|\(.*?\)|\+.*", "", name)
    name = re.sub(r"(기획|세트|한정|증정|올영픽|프로모션)", "", name)
    return name.strip()


def extract_category(name):
    for c in ["토너", "크림", "로션", "앰플", "세럼", "미스트", "패드", "클렌징", "에센스", "마스크"]:
        if c in name:
            return c
    return "기타"


def scroll_down(driver):
    """페이지를 끝까지 스크롤해서 동적 로딩 유도"""
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height


def extract_top_ingredients(driver, url, n=NUM_INGREDIENTS):
    """
    상세 페이지에서 전성분 텍스트를 추출하고, 앞쪽 주요 성분 n개 반환
    """
    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "body")))
        time.sleep(1.5)
        scroll_down(driver)

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        text = soup.get_text(" ", strip=True)

        # 전성분 섹션 찾기
        match = re.search(r"전성분[:：]?\s*(.+?)(사용법|기능성|제품정보|용량|제조사|보관방법|주의사항|$)", text)
        if not match:
            return "전성분 정보 없음"

        content = match.group(1)

        # 쉼표, 슬래시 등으로 성분 분리
        ingredients = re.split(r"[,/·•\n]", content)
        # 깨끗하게 정리 (한글, 영문, 숫자, 괄호만 남김)
        cleaned = [re.sub(r"[^가-힣a-zA-Z0-9\(\)\s]", "", ing).strip() for ing in ingredients]
        # 너무 짧은 것 제거
        cleaned = [ing for ing in cleaned if len(ing) > 1]

        top = ", ".join(cleaned[:n])
        return top if top else "전성분 정보 없음"
    except Exception as e:
        return f"오류: {e}"


def main():
    driver = start_driver()
    try:
        print("📂 스킨케어 목록 로드 중...")
        driver.get(TARGET_URL)
        WebDriverWait(driver, WAIT_TIME).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".prd_name")))
        time.sleep(2)

        html = driver.page_source
        items = parse_products(html)

        skincare_keywords = ["스킨", "토너", "로션", "크림", "앰플", "세럼", "패드", "미스트"]
        skincare_items = [i for i in items if any(k in i["name"] for k in skincare_keywords)][:10]

        enriched = []
        for idx, item in enumerate(skincare_items, start=1):
            print(f"({idx}/10) {item['name']} → 주요성분 추출 중...")
            clean_name = clean_product_name(item["name"])
            category = extract_category(clean_name)
            main_ing = extract_top_ingredients(driver, item["link"])
            enriched.append({
                "product_name": clean_name,
                "category": category,
                "main_ingredients": main_ing,
                "link": item["link"]
            })

        df = pd.DataFrame(enriched)
        df.to_excel(OUTPUT_FILE, index=False)
        print(f"\n✅ 엑셀 저장 완료: {os.path.abspath(OUTPUT_FILE)}")
        print(df)
    finally:
        driver.quit()
        print("브라우저 종료 완료.")


if __name__ == "__main__":
    main()


📂 스킨케어 목록 로드 중...
(1/10) 바이오더마[11월 올영픽] 바이오더마 하이드라비오 토너 500ml 기획(+화장솜 20매 증정) → 주요성분 추출 중...
(2/10) 브링그린[블프|일주일특가] 피지쓱싹 브링그린 티트리시카수딩토너 250mL/500mL 기획 → 주요성분 추출 중...
(3/10) 라네즈[미스트 기획/화잘먹] 라네즈 크림스킨 170ml 리필기획 (+170ml 리필+50ml+미스트펌프) → 주요성분 추출 중...
(4/10) 아누아[대용량 기획] 아누아 어성초 77 수딩 토너 350ml 기획 (+350ml 리필팩) → 주요성분 추출 중...
(5/10) 넘버즈인[모공개선/탄력광채] 넘버즈인 3번 결광가득 에센스 토너 300ml 대용량 기획 → 주요성분 추출 중...
(6/10) 라운드랩[한정] 라운드랩 1025 독도 토너 300ml 기획 (+100ml+늘어나는 스킨 패드 30매) → 주요성분 추출 중...
(7/10) 웰라쥬[첫수분토너/대용량] 웰라쥬 리얼 히알루로닉 100 토너 300ml 기획 (+화장솜 60매) → 주요성분 추출 중...
(8/10) 넘버즈인[쿨링진정]넘버즈인 1번 진정 맑게담은 청초토너 300ml 리필기획(+300ml 증정) → 주요성분 추출 중...
(9/10) 라로슈포제라로슈포제 시카플라스트 로션 B5 판테놀 시카에센스 토너 200ml → 주요성분 추출 중...
(10/10) 에스네이처[수분진정/화해1위] 에스네이처 아쿠아 오아시스 토너 300ml 기획 (+수분크림 30ml) → 주요성분 추출 중...

✅ 엑셀 저장 완료: C:\Users\UserK\oliveyoung_top_ingredients.xlsx
                                 product_name category main_ingredients  \
0                 바이오더마 바이오더마 하이드라비오 토너 500ml       토너        전성분 정보 없음   
1        브링그린 피지쓱싹 브링그린

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time, re, os

# ✅ 크롤링할 카테고리 URL
CATEGORY_URL = "https://www.oliveyoung.co.kr/store/display/getCategoryList.do?dispCatNo=10000010001"  # 스킨케어 예시

OUTPUT_NAME = "skincare_top20.xlsx"
TOP_N = 20


def start_driver():
    options = Options()
    options.add_argument("start-maximized")
    options.add_argument("disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    )
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def parse_products(html):
    soup = BeautifulSoup(html, "html.parser")
    li = soup.select("ul.cate_prd_list li")
    results = []

    for p in li:
        name = p.select_one(".prd_name")
        link = p.select_one("a[href]")

        if not name or not link:
            continue

        title = name.get_text(strip=True)
        url = link["href"]
        if url.startswith("/"):
            url = "https://www.oliveyoung.co.kr" + url

        results.append({"name": title, "link": url})
    return results


def extract_top2_ingredients(driver, url):
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "body"))
        )
        time.sleep(2)

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        text = soup.get_text(" ", strip=True)

        match = re.search(r"전성분[:：]?\s*(.+?)(사용법|주의사항|제품정보|$)", text)
        if not match:
            return "전성분 없음"

        ing = match.group(1)
        ing_list = [
            re.sub(r"[^가-힣a-zA-Z0-9() ]", "", i).strip()
            for i in re.split(r"[,/•·]", ing)
            if len(i.strip()) > 1
        ]

        return ", ".join(ing_list[:2]) if ing_list else "전성분 없음"

    except Exception:
        return "전성분 없음"


def main():
    driver = start_driver()

    try:
        print("카테고리 페이지 접속 중...")
        driver.get(CATEGORY_URL)

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".prd_name"))
        )
        time.sleep(2)

        items = parse_products(driver.page_source)[:TOP_N]

        results = []
        for idx, item in enumerate(items, 1):
            print(f"({idx}/{TOP_N}) {item['name']} 전성분 추출중...")
            top2 = extract_top2_ingredients(driver, item["link"])

            results.append({
                "product_name": item["name"],
                "top2_ingredients": top2,
                "link": item["link"]
            })

        df = pd.DataFrame(results)
        df.to_excel(OUTPUT_NAME, index=False)
        print(f"\n✅ 저장 완료 → {os.path.abspath(OUTPUT_NAME)}")

    finally:
        driver.quit()
        print("브라우저 종료 완료")


if __name__ == "__main__":
    main()


카테고리 페이지 접속 중...
브라우저 종료 완료


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=142.0.7444.61); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x9f58d3
	0x9f5914
	0x80e76d
	0x7fd980
	0x81c734
	0x8834c5
	0x89e799
	0x87c766
	0x84dac0
	0x84ede4
	0xc77974
	0xc72bea
	0xa1e5c4
	0xa0dd38
	0xa14d9d
	0x9fdee8
	0x9fe0ac
	0x9e7d2a
	0x76fd7ba9
	0x77e0c3ab
	0x77e0c32f


In [ ]:
import time, re
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


def start_driver(headless=False):
    options = Options()
    options.add_argument("start-maximized")
    options.add_argument("disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    )
    if headless:
        options.add_argument("--headless=new")

    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def parse_products(html):
    soup = BeautifulSoup(html, "html.parser")
    li = soup.select("ul.cate_prd_list li")
    results = []

    for p in li:
        name = p.select_one(".prd_name")
        link = p.select_one("a[href]")

        if not name or not link:
            continue

        title = name.get_text(strip=True)
        url = link["href"]
        if url.startswith("/"):
            url = "https://www.oliveyoung.co.kr" + url

        results.append({"name": title, "link": url})
    return results


def extract_top2_ingredients(driver, url):
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "body"))
        )
        time.sleep(2)

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        text = soup.get_text(" ", strip=True)

        match = re.search(r"전성분[:：]?\s*(.+?)(사용법|주의사항|제품정보|$)", text)
        if not match:
            return "전성분 없음"

        ing = match.group(1)
        ing_list = [
            re.sub(r"[^가-힣a-zA-Z0-9() ]", "", i).strip()
            for i in re.split(r"[,/•·]", ing)
            if len(i.strip()) > 1
        ]

        return ", ".join(ing_list[:2]) if ing_list else "전성분 없음"

    except:
        return "전성분 없음"


In [ ]:
from crawler_base import *
import pandas as pd, os

CATEGORY_URL = "https://www.oliveyoung.co.kr/store/display/getCategoryList.do?dispCatNo=10000010001"
OUTPUT = "skincare.xlsx"

TOP_N = 20

def main():
    driver = start_driver()
    driver.get(CATEGORY_URL)

    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".prd_name"))
    )
    time.sleep(2)

    items = parse_products(driver.page_source)[:TOP_N]

    rows = []
    for i, item in enumerate(items, 1):
        print(f"[{i}/{TOP_N}] {item['name']}")
        top2 = extract_top2_ingredients(driver, item["link"])

        rows.append({
            "category": "스킨케어",
            "product_name": item["name"],
            "top2_ingredients": top2,
            "link": item["link"]
        })

    pd.DataFrame(rows).to_excel(OUTPUT, index=False)
    print("저장 완료:", os.path.abspath(OUTPUT))

    driver.quit()


if __name__ == "__main__":
    main()


In [ ]:
from crawler_base import *
import pandas as pd, os

CATEGORY_URL = "https://www.oliveyoung.co.kr/store/display/getCategoryList.do?dispCatNo=10000010002"
OUTPUT = "maskpack.xlsx"
TOP_N = 20

def main():
    driver = start_driver()
    driver.get(CATEGORY_URL)

    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".prd_name"))
    )
    time.sleep(2)

    items = parse_products(driver.page_source)[:TOP_N]

    rows = []
    for i, item in enumerate(items, 1):
        print(f"[{i}/{TOP_N}] {item['name']}")
        top2 = extract_top2_ingredients(driver, item["link"])

        rows.append({
            "category": "마스크팩",
            "product_name": item["name"],
            "top2_ingredients": top2,
            "link": item["link"]
        })

    pd.DataFrame(rows).to_excel(OUTPUT, index=False)
    print("저장 완료:", os.path.abspath(OUTPUT))

    driver.quit()


if __name__ == "__main__":
    main()


In [ ]:
from crawler_base import *
import pandas as pd, os

CATEGORY_URL = "https://www.oliveyoung.co.kr/store/display/getCategoryList.do?dispCatNo=10000010003"
OUTPUT = "suncare.xlsx"
TOP_N = 20

def main():
    driver = start_driver()
    driver.get(CATEGORY_URL)

    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".prd_name"))
    )
    time.sleep(2)

    items = parse_products(driver.page_source)[:TOP_N]

    rows = []
    for i, item in enumerate(items, 1):
        print(f"[{i}/{TOP_N}] {item['name']}")
        top2 = extract_top2_ingredients(driver, item["link"])

        rows.append({
            "category": "선케어",
            "product_name": item["name"],
            "top2_ingredients": top2,
            "link": item["link"]
        })

    pd.DataFrame(rows).to_excel(OUTPUT, index=False)
    print("저장 완료:", os.path.abspath(OUTPUT))

    driver.quit()


if __name__ == "__main__":
    main()


In [ ]:
from crawler_base import *
import pandas as pd, os

CATEGORY_URL = "https://www.oliveyoung.co.kr/store/display/getCategoryList.do?dispCatNo=10000010013"
OUTPUT = "menscare.xlsx"
TOP_N = 20

def main():
    driver = start_driver()
    driver.get(CATEGORY_URL)

    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".prd_name"))
    )
    time.sleep(2)

    items = parse_products(driver.page_source)[:TOP_N]

    rows = []
    for i, item in enumerate(items, 1):
        print(f"[{i}/{TOP_N}] {item['name']}")
        top2 = extract_top2_ingredients(driver, item["link"])

        rows.append({
            "category": "맨즈케어",
            "product_name": item["name"],
            "top2_ingredients": top2,
            "link": item["link"]
        })

    pd.DataFrame(rows).to_excel(OUTPUT, index=False)
    print("저장 완료:", os.path.abspath(OUTPUT))

    driver.quit()


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd

files = {
    "스킨케어": "skincare.xlsx",
    "마스크팩": "maskpack.xlsx",
    "선케어": "suncare.xlsx",
    "맨즈케어": "menscare.xlsx"
}

writer = pd.ExcelWriter("all_categories.xlsx", engine="openpyxl")

for sheet, file in files.items():
    df = pd.read_excel(file)
    df.to_excel(writer, sheet_name=sheet, index=False)

writer.close()
print("✅ all_categories.xlsx 저장 완료!")


In [ ]:
### 서버 오토봇으로 인지하고 막힌상태 ###